# Notebook 10 — Prompt Evaluation

Build an eval pipeline with code and model-based graders.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL


## Define eval dataset

In [ ]:
EVAL_DATA = [
    {"input": "What is 2 + 2?",              "expected": "4",       "grader": "contains"},
    {"input": "Capital of Japan?",            "expected": "Tokyo",   "grader": "contains"},
    {"input": "In one word: sky colour?",     "expected": "blue",    "grader": "exact"},
    {"input": "Explain gravity in one line.", "expected": "Gravity is a force that attracts objects with mass toward each other.", "grader": "model"},
]


## Grader functions

In [ ]:
def grade_exact(actual, expected):
    return actual.strip().lower() == expected.strip().lower()

def grade_contains(actual, expected):
    return expected.strip().lower() in actual.strip().lower()

GRADER_SYSTEM = 'Reply with JSON only: {"pass": true|false, "reason": "..."}'

def grade_model(actual, expected):
    r = client.messages.create(
        model=FAST_MODEL, max_tokens=100, temperature=0.0,
        system=GRADER_SYSTEM,
        messages=[{"role": "user", "content": f"Expected: {expected}\nActual: {actual}\nDoes the actual answer satisfy the expected?"}]
    )
    try:
        result = json.loads(r.content[0].text)
        return result.get("pass", False), result.get("reason", "")
    except Exception:
        return False, r.content[0].text[:60]


## Run the eval

In [ ]:
system_under_test = "You are a concise assistant. Answer briefly."

def get_answer(q):
    r = client.messages.create(
        model=FAST_MODEL, max_tokens=200, system=system_under_test,
        messages=[{"role": "user", "content": q}]
    )
    return r.content[0].text.strip()

passed = 0
for item in EVAL_DATA:
    actual = get_answer(item["input"])
    if item["grader"] == "exact":
        ok = grade_exact(actual, item["expected"])
        reason = ""
    elif item["grader"] == "contains":
        ok = grade_contains(actual, item["expected"])
        reason = ""
    else:
        ok, reason = grade_model(actual, item["expected"])
    passed += ok
    print(f"{'✓' if ok else '✗'} [{item['grader']:<8}] {item['input'][:40]:40} | {actual[:40]}")

print(f"\nScore: {passed}/{len(EVAL_DATA)}")


## Exercise
Add 5 more test cases of your own and extend at least one grader.

In [ ]:
# Your code here
